# Experimentation: Tuning BM25 and BM25+Model1 on stackoverflow_all

This notebook runs fusion tuning on the unified collection `stackoverflow_all`.

It is configured to:
1. Train fusion parameters on `train`.
2. Evaluate on `dev1`.
3. Use the unified Lucene/forward indexes from `stackoverflow_all`.
4. Reuse an existing IBM Model 1 trained on `stackoverflow_tran` (copied into `stackoverflow_all` if needed).

### Setup
1. Set `COLLECT_ROOT`.
2. Move to `flexneuart_scripts`.
3. Use `stackoverflow_all` for train/dev1 tuning.
4. Optionally copy pre-trained Model1 files from `stackoverflow_tran`.
5. Choose candidate provider mode: `lucene`, `nmslib_bruteforce`, or `nmslib_napp`.

In [26]:
import json
import os
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
SCRIPTS_DIR = REPO_ROOT / 'demo' / 'flexneuart_scripts'
COLLECT_ROOT = (REPO_ROOT / 'demo' / 'collections').resolve()

COLLECT_NAME = 'stackoverflow_all'
MODEL1_SOURCE_COLLECTION = 'stackoverflow_tran'
MODEL1_SUBDIR = 'giza'
AUTO_COPY_MODEL1_IF_MISSING = True

TRAIN_PART = 'train'
TEST_PART = 'dev1'
TRAIN_CAND_QTY = 15

# Field choices used in your StackOverflow conversion/indexing setup
BM25_INDEX_FIELD = 'text'
BM25_QUERY_FIELD = 'text'
MODEL1_INDEX_FIELD = 'text'
MODEL1_QUERY_FIELD = 'text'

# Candidate provider mode for training/testing runs in this notebook:
#   'lucene'            -> local lucene index
#   'nmslib_bruteforce' -> NMSLIB query server configured in brute-force mode
#   'nmslib_napp'       -> NMSLIB query server configured with NAPP index
PROVIDER_MODE = 'lucene'

# Used only when PROVIDER_MODE != 'lucene'
NMSLIB_URI = 'localhost:9090'
# Path is collection-relative and points to a provider JSON with query-time params
NMSLIB_BRUTEFORCE_CONF = 'exper_desc.best/nmslib/bruteforce/cand_prov.json'
NMSLIB_NAPP_CONF = 'exper_desc.best/nmslib/napp/cand_prov.json'

os.environ['COLLECT_ROOT'] = str(COLLECT_ROOT)

collect_dir = COLLECT_ROOT / COLLECT_NAME
exper_desc_dir = collect_dir / 'exper_desc'
exper_desc_dir.mkdir(parents=True, exist_ok=True)

def provider_spec(mode: str):
    if mode == 'lucene':
        return None
    if mode == 'nmslib_bruteforce':
        return {
            'candProv': 'nmslib',
            'candProvURI': NMSLIB_URI,
            'candProvAddConf': NMSLIB_BRUTEFORCE_CONF
        }
    if mode == 'nmslib_napp':
        return {
            'candProv': 'nmslib',
            'candProvURI': NMSLIB_URI,
            'candProvAddConf': NMSLIB_NAPP_CONF
        }
    raise ValueError(f'Unknown PROVIDER_MODE: {mode}')

PROVIDER_SPEC = provider_spec(PROVIDER_MODE)
PROVIDER_CONF_REL = None if PROVIDER_SPEC is None else PROVIDER_SPEC['candProvAddConf']
PROVIDER_CONF_ABS = None if PROVIDER_CONF_REL is None else (collect_dir / PROVIDER_CONF_REL)

def apply_provider_to_descriptor(desc_rel_path: str):
    # run_experiments.sh picks candProv* from descriptor JSON, not from CLI flags.
    desc_abs = collect_dir / desc_rel_path
    with desc_abs.open('r', encoding='utf-8') as f:
        conf = json.load(f)

    if not isinstance(conf, list):
        raise RuntimeError(f'Unexpected descriptor format in {desc_abs}: expected a list')

    for entry in conf:
        if PROVIDER_SPEC is None:
            entry.pop('candProv', None)
            entry.pop('candProvURI', None)
            entry.pop('candProvAddConf', None)
        else:
            entry.update(PROVIDER_SPEC)

    with desc_abs.open('w', encoding='utf-8') as f:
        json.dump(conf, f, indent=2)

    print('Applied provider to descriptor:', desc_abs)
    print('Provider spec:', PROVIDER_SPEC)

print('REPO_ROOT      =', REPO_ROOT)
print('SCRIPTS_DIR    =', SCRIPTS_DIR)
print('COLLECT_ROOT   =', COLLECT_ROOT)
print('COLLECT_NAME   =', COLLECT_NAME)
print('TRAIN_PART     =', TRAIN_PART)
print('TEST_PART      =', TEST_PART)
print('TRAIN_CAND_QTY =', TRAIN_CAND_QTY)
print('MODEL1_SOURCE_COLLECTION =', MODEL1_SOURCE_COLLECTION)
print('MODEL1_SUBDIR  =', MODEL1_SUBDIR)
print('PROVIDER_MODE  =', PROVIDER_MODE)
print('PROVIDER_SPEC  =', PROVIDER_SPEC)
print('PROVIDER_CONF_REL =', PROVIDER_CONF_REL)
print('PROVIDER_CONF_ABS =', PROVIDER_CONF_ABS)
print('EXPER_DESC_DIR =', exper_desc_dir)

REPO_ROOT      = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
SCRIPTS_DIR    = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/flexneuart_scripts
COLLECT_ROOT   = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
COLLECT_NAME   = stackoverflow_all
TRAIN_PART     = train
TEST_PART      = dev1
TRAIN_CAND_QTY = 15
MODEL1_SOURCE_COLLECTION = stackoverflow_tran
MODEL1_SUBDIR  = giza
PROVIDER_MODE  = lucene
PROVIDER_SPEC  = None
PROVIDER_CONF_REL = None
PROVIDER_CONF_ABS = None
EXPER_DESC_DIR = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc


## 1) Ensure Model1 files exist in stackoverflow_all

`run_experiments.sh` expects Model1 files under the same collection being tuned.

If Model1 is missing in `stackoverflow_all`, this cell copies `derived_data/giza` from `stackoverflow_tran`.

In [27]:
target_model1_dir = collect_dir / 'derived_data' / MODEL1_SUBDIR
target_model1_file = target_model1_dir / MODEL1_INDEX_FIELD / 'output.t1.5.bin'

src_model1_dir = COLLECT_ROOT / MODEL1_SOURCE_COLLECTION / 'derived_data' / MODEL1_SUBDIR
src_model1_file = src_model1_dir / MODEL1_INDEX_FIELD / 'output.t1.5.bin'

print('Target Model1 dir :', target_model1_dir)
print('Target Model1 file:', target_model1_file)
print('Source Model1 dir :', src_model1_dir)
print('Source Model1 file:', src_model1_file)

if not target_model1_file.exists():
    if not AUTO_COPY_MODEL1_IF_MISSING:
        raise FileNotFoundError(
            f'Model1 missing in target collection: {target_model1_file}. '
            'Set AUTO_COPY_MODEL1_IF_MISSING=True or copy manually.'
        )

    if not src_model1_file.exists():
        raise FileNotFoundError(
            f'Source Model1 file does not exist: {src_model1_file}. '
            'Please train Model1 first or fix MODEL1_SOURCE_COLLECTION/MODEL1_INDEX_FIELD.'
        )

    target_model1_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_model1_dir, target_model1_dir, dirs_exist_ok=True)
    print('Copied Model1 directory to target collection.')

if not target_model1_file.exists():
    raise FileNotFoundError(f'Expected Model1 file after copy: {target_model1_file}')

print('Model1 is available for stackoverflow_all.')

Target Model1 dir : /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/derived_data/giza
Target Model1 file: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/derived_data/giza/text/output.t1.5.bin
Source Model1 dir : /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza
Source Model1 file: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_tran/derived_data/giza/text/output.t1.5.bin
Model1 is available for stackoverflow_all.


## 2) Validate prerequisites

Checks `train/dev1` input data, unified indexes, and Model1 files in `stackoverflow_all`.

In [28]:
required_paths = [
    collect_dir / 'input_data' / TRAIN_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TRAIN_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TRAIN_PART / 'qrels.txt',
    collect_dir / 'input_data' / TEST_PART / 'QuestionFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'AnswerFields.jsonl',
    collect_dir / 'input_data' / TEST_PART / 'qrels.txt',
    collect_dir / 'lucene_index',
    collect_dir / 'forward_index' / BM25_INDEX_FIELD,
    collect_dir / 'derived_data' / MODEL1_SUBDIR / MODEL1_INDEX_FIELD / 'output.t1.5.bin'
 ]

for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f'Missing prerequisite: {p}')

if PROVIDER_MODE != 'lucene':
    if PROVIDER_CONF_ABS is None or not PROVIDER_CONF_ABS.exists():
        raise FileNotFoundError(
            f'NMSLIB provider config is missing: {PROVIDER_CONF_ABS}. '
            "Either create this JSON file, or set PROVIDER_MODE='lucene' in the setup cell."
        )
    print('Found NMSLIB provider config:', PROVIDER_CONF_ABS)

print('All prerequisites found for stackoverflow_all.')

All prerequisites found for stackoverflow_all.


## 3) Tuning BM25
Generate BM25 tuning descriptors (`text` vs `text`).

In [29]:
cmd = [
    'python3', './gen_exper_desc/gen_bm25_tune_json_desc.py',
    '--index_field_name', BM25_INDEX_FIELD,
    '--query_field_name', BM25_QUERY_FIELD,
    '--outdir', str(exper_desc_dir),
    '--exper_subdir', 'tuning',
    '--rel_desc_path', 'exper_desc'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'BM25 descriptor generation failed: {res.returncode}')

Running: python3 ./gen_exper_desc/gen_bm25_tune_json_desc.py --index_field_name text --query_field_name text --outdir /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc --exper_subdir tuning --rel_desc_path exper_desc
Namespace(outdir='/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc', rel_desc_path='exper_desc', exper_subdir='tuning', index_field_name='text', query_field_name='text', cand_prov_uri=None, cand_prov_add_conf=None, cand_prov_qty=None)



In [ ]:
bm25_desc = f'exper_desc/bm25tune_{BM25_QUERY_FIELD}_{BM25_INDEX_FIELD}.json'
apply_provider_to_descriptor(bm25_desc)

cmd = [
    'bash', './exper/run_experiments.sh',
    COLLECT_NAME,
    bm25_desc,
    '-test_part', TEST_PART,
    '-train_part', TRAIN_PART,
    '-train_cand_qty', str(TRAIN_CAND_QTY),
    '-model1_subdir', MODEL1_SUBDIR
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'BM25 tuning experiments failed: {res.returncode}')

Applied provider to descriptor: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_all/exper_desc/bm25tune_text_text.json
Provider spec: None
Running: bash ./exper/run_experiments.sh stackoverflow_all exper_desc/bm25tune_text_text.json -test_part dev1 -train_part train -train_cand_qty 15 -model1_subdir giza


In [31]:
bm25_desc = f'exper_desc/bm25tune_{BM25_QUERY_FIELD}_{BM25_INDEX_FIELD}.json'
cmd = [
    'bash', './report/get_exper_results.sh',
    COLLECT_NAME,
    bm25_desc,
    'bm25tune_stackoverflow_all.tsv',
    '-test_part', TEST_PART,
    '-flt_cand_qty', '250',
    '-print_best_metr', 'map'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'BM25 results extraction failed: {res.returncode}')

Running: bash ./report/get_exper_results.sh stackoverflow_all exper_desc/bm25tune_text_text.json bm25tune_stackoverflow_all.tsv -test_part dev1 -flt_cand_qty 250 -print_best_metr map
Using collection root: /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections
Including only runs that generated 250 candidate records
Best results for metric map:
Value: 0.014200
Result sub-dir: tuning/bm25tune_text_text/bm25tune_k1=0.4_b=0.3



## 4) Tuning a Fusion of Model1 and BM25
Set `BEST_BM25_K1` and `BEST_BM25_B` from `bm25tune_stackoverflow_all.tsv`.

In [ ]:
# Replace with best BM25 params from bm25tune_stackoverflow_all.tsv
BEST_BM25_K1 = 0.6
BEST_BM25_B = 0.3

cmd = [
    'python3', './gen_exper_desc/gen_model1_exper_json_desc.py',
    '-k1', str(BEST_BM25_K1),
    '-b', str(BEST_BM25_B),
    '--exper_subdir', 'tuning',
    '--query_field_name', MODEL1_QUERY_FIELD,
    '--index_field_name', MODEL1_INDEX_FIELD,
    '--outdir', str(exper_desc_dir),
    '--rel_desc_path', 'exper_desc'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Model1+BM25 descriptor generation failed: {res.returncode}')

In [ ]:
model1_desc = f'exper_desc/model1tune_{MODEL1_QUERY_FIELD}_{MODEL1_INDEX_FIELD}.json'
apply_provider_to_descriptor(model1_desc)

cmd = [
    'bash', './exper/run_experiments.sh',
    COLLECT_NAME,
    model1_desc,
    '-test_part', TEST_PART,
    '-train_part', TRAIN_PART,
    '-train_cand_qty', str(TRAIN_CAND_QTY),
    '-model1_subdir', MODEL1_SUBDIR
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Model1+BM25 tuning experiments failed: {res.returncode}')

In [ ]:
model1_desc = f'exper_desc/model1tune_{MODEL1_QUERY_FIELD}_{MODEL1_INDEX_FIELD}.json'
cmd = [
    'bash', './report/get_exper_results.sh',
    COLLECT_NAME,
    model1_desc,
    'model1tune_stackoverflow_all.tsv',
    '-test_part', TEST_PART,
    '-flt_cand_qty', '250',
    '-print_best_metr', 'map'
]
print('Running:', ' '.join(cmd))
res = subprocess.run(cmd, cwd=SCRIPTS_DIR, env=os.environ.copy(), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Model1+BM25 results extraction failed: {res.returncode}')

### Notes
- `TRAIN_PART = train` and `TEST_PART = dev1` are set for fair tuning/validation.
- This notebook uses indexes from `stackoverflow_all` (Lucene + forward).
- If Model1 is not already in `stackoverflow_all/derived_data/giza`, it is copied from `stackoverflow_tran`.
- Outputs are written to `bm25tune_stackoverflow_all.tsv` and `model1tune_stackoverflow_all.tsv` in `demo/flexneuart_scripts`.
- Provider selection is controlled by `PROVIDER_MODE` in Cell 3: `lucene`, `nmslib_bruteforce`, or `nmslib_napp`.
- Provider mode is applied by patching descriptor JSON entries (`candProv`, `candProvURI`, `candProvAddConf`) before each run.
- For `nmslib_*` modes, you must start an NMSLIB query server and make sure `NMSLIB_URI` plus the corresponding `cand_prov.json` path are valid.